# 5-fold cross-validation for the fixed-weight SSL+rebalance model (Correction 1, part 2)


Needs GPU. Pools `expA_train.csv` + `expA_val.csv` (587 images) into 5 stratified folds; `expA_test.csv` (104 images) stays untouched and is used only for final evaluation. Requires the Stage-1 SSL-pretrained encoder checkpoint. Edit the three `EDIT THESE` paths below.

In [1]:
# ============================================================
# 5-fold stratified cross-validation for the SSL + rebalance
# (fixed-weight) pipeline, to get a distribution-based variance
# estimate for MEL recall beyond the existing 3-seed report.
#
# DATASETS NEEDED (edit the paths below):
#   - Pretrained SSL encoder weights from Stage 1 (SimCLR
#     pretraining on the 24,000 unlabeled ISIC images) -- point
#     SSL_ENCODER_CKPT at that .pth file. This is the Stage-1
#     encoder only, NOT one of the finetuned checkpoints.
#   - expA_train.csv and expA_val.csv (same CSVs used elsewhere)
#     -- these get POOLED into one 587-image set and re-split
#     into 5 stratified folds. Do NOT use expA_test.csv here; it
#     stays completely untouched and is used only for final
#     evaluation, exactly as in your original fixed-weight run.
#   - The ISIC image folder (LABELED_DIR), same as always.
#
# WHAT THIS DOES:
#   For each of 5 stratified folds (by label), 4/5 of the pooled
#   587 images become the fine-tuning training set and 1/5
#   becomes the checkpoint-selection validation set for that
#   fold. The SSL-pretrained encoder is loaded fresh per fold
#   (not re-pretrained), then fully fine-tuned with the same
#   recipe as your main result: 5x MEL oversampling, heavy MEL
#   augmentation, focal loss (gamma=1.5, weights 4/2/1), Adam
#   1e-4, batch 16, grad clip 1.0, early stopping patience 7,
#   checkpoint selected by validation macro-F1.
#
#   After each fold's model is selected, it is evaluated on the
#   ORIGINAL, unchanged 104-image test set (expA_test.csv) -- the
#   same test set used in every other result in the paper -- so
#   the 5 fold results are directly comparable to the existing
#   3-seed numbers.
#
# OUTPUT:
#   - kfold_results.csv: one row per fold with accuracy,
#     macro-F1, MEL recall, MEL precision on the held-out test set.
#   - Printed mean +/- SD across the 5 folds, to report as
#     "5-fold CV: MEL recall = X% +/- Y%" alongside the existing
#     3-seed number in the paper.
#
# NOTE: Set NUM_SEEDS > 1 to repeat the whole 5-fold procedure
# with different random inits per fold (NUM_SEEDS x 5 total
# runs) for an even more robust estimate. Default is 1 seed
# (5 runs total) to keep this affordable; bump to 3 if you have
# GPU budget for ~15 runs.
# ============================================================
import os, glob, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- EDIT THESE ----
BACKUP_DIR       = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
LABELED_DIR      = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
SSL_ENCODER_CKPT = '/kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth'
# ---------------------

OUTPUT_DIR = '/kaggle/working'
N_FOLDS = 5
NUM_SEEDS = 1          # bump to 3 for a NUM_SEEDS x 5-fold estimate if GPU budget allows
CLASSES = ['BKL', 'MEL', 'NV']
MEL_OVERSAMPLE = 5
MAX_EPOCHS = 30
PATIENCE = 7

train_transform_std = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_transform_mel = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45), transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def resolve_path(img_name):
    if not img_name.endswith('.jpg'):
        img_name += '.jpg'
    p = os.path.join(LABELED_DIR, img_name)
    if not os.path.exists(p):
        m = glob.glob(f"{LABELED_DIR}/**/{img_name}", recursive=True)
        p = m[0] if m else p
    return p


class LesionDataset(Dataset):
    def __init__(self, df, training):
        self.df = df.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(resolve_path(str(row['image']))).convert('RGB')
        tfm = (train_transform_mel if row['label'] == 'MEL' else train_transform_std) if self.training else eval_transform
        return tfm(img), row['label']


def make_oversampled_df(df):
    mel = df[df['label'] == 'MEL']
    rest = df[df['label'] != 'MEL']
    return pd.concat([rest] + [mel] * MEL_OVERSAMPLE, ignore_index=True)


class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )

    def forward(self, x):
        return self.features(x)


class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.encoder(x))


def load_encoder_ckpt(encoder, ckpt_path):
    """Robust to a checkpoint saved either as a bare SimpleEncoder
    state_dict or as an SSLClassifier state_dict with an 'encoder.'
    prefix -- strips the prefix if present, and reports any keys
    that still don't line up instead of crashing silently."""
    state = torch.load(ckpt_path, map_location=device)
    stripped = {(k[len('encoder.'):] if k.startswith('encoder.') else k): v for k, v in state.items()}
    missing, unexpected = encoder.load_state_dict(stripped, strict=False)
    if missing or unexpected:
        print(f"  [warning] encoder load: {len(missing)} missing, {len(unexpected)} unexpected keys "
              f"-- verify SSL_ENCODER_CKPT points at the right file.")
    return encoder


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=1.5):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        a_t = self.alpha[targets]
        return (-a_t * (1 - p_t) ** self.gamma * logp_t).mean()


def evaluate(model, loader):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1).cpu().numpy()
            all_pred.extend([CLASSES[p] for p in preds])
            all_true.extend(labels)
    true_arr, pred_arr = np.array(all_true), np.array(all_pred)
    acc = np.mean(pred_arr == true_arr)
    macro_f1 = f1_score(true_arr, pred_arr, labels=CLASSES, average='macro', zero_division=0)
    tp = np.sum((true_arr == 'MEL') & (pred_arr == 'MEL'))
    fn = np.sum((true_arr == 'MEL') & (pred_arr != 'MEL'))
    fp = np.sum((true_arr != 'MEL') & (pred_arr == 'MEL'))
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return acc, macro_f1, recall, precision


def train_one_fold(train_df, val_df, test_df, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    class_to_idx = {c: i for i, c in enumerate(CLASSES)}

    train_loader = DataLoader(LesionDataset(make_oversampled_df(train_df), training=True), batch_size=16, shuffle=True)
    val_loader = DataLoader(LesionDataset(val_df, training=False), batch_size=16, shuffle=False)
    test_loader = DataLoader(LesionDataset(test_df, training=False), batch_size=16, shuffle=False)

    encoder = SimpleEncoder()
    encoder = load_encoder_ckpt(encoder, SSL_ENCODER_CKPT)
    model = SSLClassifier(encoder, len(CLASSES)).to(device)

    alpha = torch.tensor([2.0, 4.0, 1.0], device=device)  # BKL, MEL, NV order per CLASSES
    criterion = FocalLoss(alpha, gamma=1.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            targets = torch.tensor([class_to_idx[l] for l in labels], device=device)
            optimizer.zero_grad()
            loss = criterion(model(images), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        _, val_f1, _, _ = evaluate(model, val_loader)
        if val_f1 > best_f1:
            best_f1, best_state, no_improve = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return evaluate(model, test_loader)


train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df_orig = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
pool_df = pd.concat([train_df, val_df_orig], ignore_index=True)
print(f"Pooled train+val: {len(pool_df)} images ({pool_df['label'].value_counts().to_dict()})")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
results = []

NUM_SEEDS = 3
for seed_i in range(NUM_SEEDS):
    seed = [42, 123, 2024][seed_i % 3]
    for fold_i, (tr_idx, va_idx) in enumerate(skf.split(pool_df, pool_df['label'])):
        fold_train = pool_df.iloc[tr_idx]
        fold_val = pool_df.iloc[va_idx]
        acc, f1_, rec, prec = train_one_fold(fold_train, fold_val, test_df, seed)
        print(f"[seed {seed}] fold {fold_i}: acc={acc:.3f} macroF1={f1_:.3f} MELrec={rec:.3f} MELprec={prec:.3f}")
        results.append({'seed': seed, 'fold': fold_i, 'accuracy': acc, 'macro_f1': f1_,
                         'mel_recall': rec, 'mel_precision': prec})

results_df = pd.DataFrame(results)
out_path = f'{OUTPUT_DIR}/kfold_results.csv'
results_df.to_csv(out_path, index=False)
print(f"\nWrote {out_path}")
print("\nMean +/- SD across all runs (report this alongside the existing 3-seed number):")
for metric in ['accuracy', 'macro_f1', 'mel_recall', 'mel_precision']:
    print(f"  {metric:15s}: {results_df[metric].mean():.3f} +/- {results_df[metric].std():.3f}")


Pooled train+val: 587 images ({'NV': 425, 'BKL': 120, 'MEL': 42})
[seed 42] fold 0: acc=0.683 macroF1=0.526 MELrec=0.375 MELprec=0.231
[seed 42] fold 1: acc=0.663 macroF1=0.516 MELrec=0.375 MELprec=0.188
[seed 42] fold 2: acc=0.654 macroF1=0.476 MELrec=0.500 MELprec=0.200
[seed 42] fold 3: acc=0.625 macroF1=0.477 MELrec=0.500 MELprec=0.200
[seed 42] fold 4: acc=0.635 macroF1=0.500 MELrec=0.375 MELprec=0.231
[seed 123] fold 0: acc=0.673 macroF1=0.518 MELrec=0.375 MELprec=0.167
[seed 123] fold 1: acc=0.702 macroF1=0.545 MELrec=0.375 MELprec=0.176
[seed 123] fold 2: acc=0.712 macroF1=0.533 MELrec=0.375 MELprec=0.231
[seed 123] fold 3: acc=0.644 macroF1=0.508 MELrec=0.375 MELprec=0.150
[seed 123] fold 4: acc=0.721 macroF1=0.562 MELrec=0.375 MELprec=0.250
[seed 2024] fold 0: acc=0.702 macroF1=0.552 MELrec=0.375 MELprec=0.250
[seed 2024] fold 1: acc=0.702 macroF1=0.533 MELrec=0.375 MELprec=0.250
[seed 2024] fold 2: acc=0.721 macroF1=0.550 MELrec=0.375 MELprec=0.231
[seed 2024] fold 3: acc=0.